# 07 — Operator Hint Experiment (±1 VM/step, constrained)

The companion to notebook 06. Here the agent's scaling is **constrained to
±1 VM/step**, so it *cannot* react quickly to a surge — it would have to
anticipate. This tests whether advance hints become more valuable when reactive
capacity is limited.

The constraint is applied via an isolated `ConstrainedCloudEnv` subclass
defined **in this notebook only** — `env.py` is not modified, so all other
results are unaffected.

Together with notebook 06 (±5), this brackets the regime where operator hints
add value.

## 1. Imports and the isolated constrained environment

In [1]:
import json
import numpy as np
import torch

from env import CloudClusterEnv, STEPS_PER_WEEK, MIN_PODS, MAX_PODS
from agent import ActorCritic, train_ppo

stats = json.load(open('trace_params.json'))['stats']

class ConstrainedCloudEnv(CloudClusterEnv):
    """Identical to CloudClusterEnv but scaling is limited to ±1 VM/step.
    env.py is NOT modified — this override lives only in this notebook."""
    def step(self, action):
        self._update_hints()
        delta = int(round(float(action[0]) * 1))     # ±1 VM/step (was ±5)
        self.active_vms = int(np.clip(self.active_vms + delta, MIN_PODS, MAX_PODS))
        self.queue.extend(self._generate_jobs())
        jobs_processed, avg_cpu, max_cpu, avg_mem, max_mem = self._assign_jobs()
        breaches = self._check_deadlines()
        self.total_breaches += breaches
        cost = self.active_vms / MAX_PODS
        self.cost_total += cost
        utilisation = max(avg_cpu, avg_mem)
        jobs_due = jobs_processed + breaches
        breach_rate = breaches / jobs_due if jobs_due > 0 else 0.0
        reward = (- self.lambda_cost * cost - self.lambda_sla * breach_rate
                  + self.lambda_util * utilisation)
        self.history.append([avg_cpu, max_cpu, avg_mem, max_mem]); self.history.pop(0)
        self.prev_queue_len = len(self.queue); self.step_count += 1
        obs = self._build_state()
        done = self.step_count >= STEPS_PER_WEEK
        info = {'cost': cost, 'breaches': breaches, 'utilisation': utilisation,
                'active_vms': self.active_vms, 'queue': len(self.queue),
                'avg_cpu': avg_cpu, 'avg_mem': avg_mem, 'hint_active': self.hint_active}
        return obs, reward, done, False, info

print("ConstrainedCloudEnv (±1 VM/step) defined — isolated.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
ConstrainedCloudEnv (±1 VM/step) defined — isolated.


## 2. Train a hint-aware agent under the ±1 constraint

**~15-20 min run.**

In [2]:
env_c = ConstrainedCloudEnv(stats, enable_surges=True, enable_hints=True,
                            min_surges=8, max_surges=12, seed=None)
net_c = ActorCritic()
optimizer = torch.optim.Adam(net_c.parameters(), lr=3e-4)

print("Training constrained hint-aware agent (±1 VM/step)...\n")
episode_rewards, convergence_log = train_ppo(env_c, net_c, optimizer,
                                             total_steps=250_000)

torch.save(net_c.state_dict(), 'ppo_hint_constrained.pth')
json.dump(convergence_log, open('ppo_hint_constrained_convergence.json', 'w'))
print(f"\nSaved. First: {episode_rewards[0]:.1f}  Last: {episode_rewards[-1]:.1f}")

Training constrained hint-aware agent (±1 VM/step)...

steps   2048 | recent ep reward   -203.8
steps   4096 | recent ep reward   -201.5
steps   6144 | recent ep reward   -195.7
steps   8192 | recent ep reward   -195.3
steps  10240 | recent ep reward   -195.2
steps  12288 | recent ep reward   -186.4
steps  14336 | recent ep reward   -213.1
steps  16384 | recent ep reward   -203.2
steps  18432 | recent ep reward   -202.1
steps  20480 | recent ep reward   -178.3
steps  22528 | recent ep reward   -189.7
steps  24576 | recent ep reward   -212.6
steps  26624 | recent ep reward   -200.3
steps  28672 | recent ep reward   -191.2
steps  30720 | recent ep reward   -212.7
steps  32768 | recent ep reward   -203.6
steps  34816 | recent ep reward   -204.5
steps  36864 | recent ep reward   -217.4
steps  38912 | recent ep reward   -227.8
steps  40960 | recent ep reward   -236.4
steps  43008 | recent ep reward   -240.1
steps  45056 | recent ep reward   -236.7
steps  47104 | recent ep reward   -208.4
st

## 3. Evaluate: with hints vs hints-blanked (constrained, same surges)

In [3]:
def eval_constrained(net, n_episodes, force_zero_hints, seed_base=5000):
    breaches_list = []
    for i in range(n_episodes):
        e = ConstrainedCloudEnv(stats, enable_surges=True, enable_hints=True,
                                min_surges=8, max_surges=12, seed=seed_base+i)
        obs, _ = e.reset()
        obs = torch.tensor(obs, dtype=torch.float32)
        ep = 0
        for t in range(STEPS_PER_WEEK):
            if force_zero_hints:
                obs[-3:] = 0.0
            with torch.no_grad():
                mean, _ = net.forward(obs.unsqueeze(0))
            obs, r, done, tr, info = e.step(mean.squeeze(0).numpy())
            obs = torch.tensor(obs, dtype=torch.float32)
            ep += info['breaches']
            if done: break
        breaches_list.append(ep)
    return float(np.mean(breaches_list)), float(np.std(breaches_list))

net_c = ActorCritic()
net_c.load_state_dict(torch.load('ppo_hint_constrained.pth'))
net_c.eval()

N = 30
with_h, with_std       = eval_constrained(net_c, N, force_zero_hints=False)
without_h, without_std = eval_constrained(net_c, N, force_zero_hints=True)

reduction = (without_h - with_h) / without_h * 100 if without_h > 0 else 0.0

print("="*54)
print("HINT EXPERIMENT — ±1 VM/step constrained (30 weeks)")
print("="*54)
print(f"  WITH hints active:       {with_h:>8.0f} ± {with_std:.0f} breaches/week")
print(f"  WITHOUT hints (blanked): {without_h:>8.0f} ± {without_std:.0f} breaches/week")
print("="*54)
print(f"\n  Hints reduce breaches by {reduction:.1f}%")

json.dump({'regime': '±1 VM/step', 'with_hints': with_h, 'without_hints': without_h,
           'with_std': with_std, 'without_std': without_std, 'reduction_pct': reduction},
          open('hint_result_constrained.json', 'w'), indent=2)
print("Saved hint_result_constrained.json")

HINT EXPERIMENT — ±1 VM/step constrained (30 weeks)
  WITH hints active:          76986 ± 19330 breaches/week
  WITHOUT hints (blanked):    76986 ± 19330 breaches/week

  Hints reduce breaches by 0.0%
Saved hint_result_constrained.json
